### Connect postgresql database

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# 数据库配置
username = "XXXXXX"
password = "YYYYYY"
host = "localhost"
port = 5432
database = "eyewear-data"

# 创建连接
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

# 查询数据
sql = """
SELECT
    o.order_id,
    o.customer_id,
    o.order_date,
    o.order_status,
    o.total_price_before_tax,
    oi.product_id,
    oi.quantity,
    oi.product_name,
    oi.unit_price,
    oi.line_price_before_tax,
    oi.is_free_gift,
    pa.campaign_id,
    ci.customer_hierarchy,
    ci.customer_type,
    ci.channel_source, 
    pi.category,
    pi.sub_category,
    pi.brand,
    si.store_name,
    si.store_type,
    si.store_city,
    si.store_region,
    si.store_country,
    si.latitude,
    si.longitude
FROM "Order" o
LEFT JOIN "OrderItem" oi
    ON o.order_id = oi.order_id
LEFT JOIN "PromotionActivity" pa
    ON o.campaign_id = pa.campaign_id
LEFT JOIN "ProductInfo" pi
    ON oi.product_id = pi.product_id
LEFT JOIN "CustomerInfo" ci
    ON o.customer_id = ci.customer_id
LEFT JOIN "StoreInfo" si
    ON ci.preferred_store_id = si.store_id
WHERE o.order_status IN ('Completed', 'Shipped')
"""

df_order_completed_shipped = pd.read_sql(sql, engine)

# 导出数据
df_order_completed_shipped.to_parquet('2023-2024_product_customer_store_type.parquet', engine='fastparquet', index=False)
print("数据已保存为 2023-2024_product_customer_store_type.parquet 文件")
# 查看数据
df_order_completed_shipped

数据已保存为 2023-2024_product_customer_store_type.parquet 文件


,order_id,customer_id,order_date,order_status,total_price_before_tax,product_id,quantity,product_name,unit_price,line_price_before_tax,...,category,sub_category,brand,store_name,store_type,store_city,store_region,store_country,latitude,longitude
0,37,157800,2023-03-25 11:47:39,Completed,317.79,105,1,Ray-Ban Flacko Prescription,188.26,150.61,...,Eyeglasses,Eyeglasses,Ray-Ban,Ray-Ban Tampa,Ray-Ban Store,Tampa,Florida,United States,27.585017,-81.528478
1,37,157800,2023-03-25 11:47:39,Completed,317.79,97,1,Ray-Ban New Wayfarer Prescription,208.97,167.18,...,Eyeglasses,Eyeglasses,Ray-Ban,Ray-Ban Tampa,Ray-Ban Store,Tampa,Florida,United States,27.585017,-81.528478
2,38,57296,2023-02-15 12:30:36,Completed,777.07,86,1,Ray-Ban Original Wayfarer Prescription,163.18,163.18,...,Eyeglasses,Eyeglasses,Ray-Ban,Ray-Ban San Diego,Ray-Ban Store,San Diego,California,United States,36.761941,-119.413001
3,38,57296,2023-02-15 12:30:36,Completed,777.07,171,2,Ray-Ban Violet Single vision,198.71,397.42,...,Lens,Violet Lens,Ray-Ban,Ray-Ban San Diego,Ray-Ban Store,San Diego,California,United States,36.761941,-119.413001
4,38,57296,2023-02-15 12:30:36,Completed,777.07,56,1,Ray-Ban Bill Non-prescription,201.47,201.47,...,Eyeglasses,Eyeglasses,Ray-Ban,Ray-Ban San Diego,Ray-Ban Store,San Diego,California,United States,36.761941,-119.413001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
655896,499985,36208,2024-10-07 12:59:34,Completed,433.24,107,1,Ray-Ban Ray-Ban Reverse Prescription,433.24,433.24,...,AI Glasses,AI Smart Glasses,Ray-Ban,LensCrafters Coronado,Multi-brand Retail,New York,New York,United States,40.644609,-74.081084
655897,499987,136134,2024-10-25 05:50:04,Completed,744.64,152,1,Ray-Ban Ray-Ban Meta Prescription,254.33,206.01,...,AI Glasses,AI Smart Glasses,Ray-Ban,Ray-Ban Millenia,Ray-Ban Store,Millenia,Florida,United States,27.606965,-81.561521
655898,499987,136134,2024-10-25 05:50:04,Completed,744.64,40,1,Ray-Ban Ray-Ban Meta Prescription,237.84,192.65,...,AI Glasses,AI Smart Glasses,Ray-Ban,Ray-Ban Millenia,Ray-Ban Store,Millenia,Florida,United States,27.606965,-81.561521
655899,499987,136134,2024-10-25 05:50:04,Completed,744.64,69,1,Ray-Ban Round Metal Non-prescription,427.14,345.98,...,Sunglasses,Classic Sunglasses,Ray-Ban,Ray-Ban Millenia,Ray-Ban Store,Millenia,Florida,United States,27.606965,-81.561521


### cal product type

In [20]:
df_product_categories = df_order_completed_shipped['category'].unique()
df_product_categories

array(['Eyeglasses', 'Lens', 'Sunglasses', 'AI Glasses', 'Accessory'],
      dtype=object)

In [21]:
df_product_sub_categories = df_order_completed_shipped['sub_category'].unique()
df_product_sub_categories

array(['Eyeglasses', 'Violet Lens', 'Classic Sunglasses',
       'Transitions Lens', 'Evolve Lens', 'AI Smart Glasses',
       'Mirror Lens', 'Polarized+ Lens', 'Gradient Lens',
       'Solid Color Lens', 'Chromance Lens', 'Polarized S Lens',
       'Clear Lens', 'Free Gift'], dtype=object)

In [22]:
df_product_brands = df_order_completed_shipped['brand'].unique()
df_product_brands

array(['Ray-Ban', 'Essilor'], dtype=object)

In [23]:
df_customer_types = df_order_completed_shipped['customer_type'].unique()
df_customer_types

array(['Promotional Sensitive Customers', 'Regular Customers',
       'Lens Customers', 'VIP Loyal Customers', 'One-time Customers'],
      dtype=object)

In [26]:
df_store_types = df_order_completed_shipped['store_type'].unique()
df_store_types

array(['Ray-Ban Store', 'Multi-brand Retail'], dtype=object)

In [27]:
df_store_regions = df_order_completed_shipped['store_region'].unique()
df_store_regions

array(['Florida', 'California', 'Illinois', 'New York', 'Ohio',
       'Massachusetts', 'Georgia', 'New Jersey', 'Texas',
       'District of Columbia', 'Colorado', 'Hawaii', 'Washington',
       'Nevada', 'Tennessee', 'North Carolina'], dtype=object)